# Experiment 2: Known-Structure Recovery (GPU-Accelerated)

**Purpose:** Verify that SuStaIn can detect genuine subtypes when they exist, validating that failure in null data reflects absence of structure rather than methodological limitation.

**Design:** Generate synthetic data with 3 pre-defined subtypes (Musculoskeletal-dominant, Autonomic-dominant, Neurological-dominant) with ground-truth Cohen's d > 1.0 for domain-defining symptoms. Run SuStaIn and assess recovery via ARI.

**Success criterion:** ARI > 0.7 comparing SuStaIn assignments to ground truth.

**Runtime:** ~5-10 min on T4 GPU (vs ~80 min on CPU)

## 0. Environment Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
!pip install -q torch numpy scipy scikit-learn pandas matplotlib seaborn tqdm tabulate

In [ ]:
# Clone FIXED repo (with GPU bug fix)
import os
if os.path.exists('mphil_repo'):
    !rm -rf mphil_repo
!git clone --depth 1 --branch claude/convert-to-jupyter-01GY8iZvAixjYs4t3VyLsWHf https://github.com/Amelia3141/mphil.git mphil_repo

import sys
sys.path.insert(0, 'mphil_repo')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import json
import tempfile
import time
import warnings
from itertools import combinations
from scipy.stats import norm
from scipy.linalg import cholesky
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score, confusion_matrix
warnings.filterwarnings('ignore')

# GPU check
if torch.cuda.is_available():
    device = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {device} ({mem:.1f} GB)')
    USE_GPU = True
else:
    print('No GPU - will be slow. Go to Runtime -> Change runtime type -> T4 GPU')
    USE_GPU = False

print('All imports successful.')

## 1. Define Constants

In [ ]:
SYMPTOM_NAMES = [
    'fatigue', 'chronic_pain', 'sleep_disturbance',
    'joint_pain', 'subluxations', 'joint_hypermobility',
    'gi_symptoms',
    'pots_dysautonomia',
    'headaches', 'cognitive_dysfunction',
    'anxiety', 'depression',
    'mcas', 'allergies',
    'skin_fragility', 'urinary_symptoms', 'tmj_dysfunction',
    'vision_issues', 'hearing_tinnitus'
]

N_BIOMARKERS = len(SYMPTOM_NAMES)
N_SCORES = 2  # 2 severity levels (1=mild, 2=severe); 0=absent handled separately

PREVALENCES = {
    'fatigue': 0.78, 'chronic_pain': 0.82, 'sleep_disturbance': 0.65,
    'joint_pain': 0.85, 'subluxations': 0.70, 'joint_hypermobility': 0.98,
    'gi_symptoms': 0.75, 'pots_dysautonomia': 0.45,
    'headaches': 0.60, 'cognitive_dysfunction': 0.55,
    'anxiety': 0.65, 'depression': 0.45,
    'mcas': 0.25, 'allergies': 0.50,
    'skin_fragility': 0.70, 'urinary_symptoms': 0.40,
    'tmj_dysfunction': 0.35, 'vision_issues': 0.30,
    'hearing_tinnitus': 0.25
}

DOMAINS = {
    'Musculoskeletal': ['joint_pain', 'subluxations', 'joint_hypermobility'],
    'Autonomic':       ['pots_dysautonomia'],
    'Gastrointestinal':['gi_symptoms'],
    'Neurological':    ['headaches', 'cognitive_dysfunction'],
    'Mental Health':   ['anxiety', 'depression'],
    'Immune/Inflam.':  ['mcas', 'allergies'],
    'Systemic':        ['fatigue', 'chronic_pain', 'sleep_disturbance']
}

SUBTYPE_NAMES_TRUE = ['Musculoskeletal', 'Autonomic', 'Neurological']

print(f'{N_BIOMARKERS} symptoms, {N_SCORES} severity levels')

In [ ]:
def build_correlation_matrix():
    n = len(SYMPTOM_NAMES)
    corr = np.eye(n)
    idx = {name: i for i, name in enumerate(SYMPTOM_NAMES)}
    def s(a, b, r):
        corr[idx[a], idx[b]] = r
        corr[idx[b], idx[a]] = r
    s('fatigue', 'chronic_pain', 0.55)
    s('fatigue', 'sleep_disturbance', 0.50)
    s('chronic_pain', 'sleep_disturbance', 0.45)
    s('anxiety', 'depression', 0.55)
    s('joint_pain', 'subluxations', 0.50)
    s('gi_symptoms', 'mcas', 0.45)
    s('pots_dysautonomia', 'fatigue', 0.45)
    s('chronic_pain', 'depression', 0.45)
    s('anxiety', 'pots_dysautonomia', 0.35)
    s('fatigue', 'cognitive_dysfunction', 0.35)
    s('mcas', 'skin_fragility', 0.30)
    s('mcas', 'allergies', 0.40)
    s('headaches', 'cognitive_dysfunction', 0.35)
    s('chronic_pain', 'anxiety', 0.30)
    s('depression', 'fatigue', 0.35)
    s('pots_dysautonomia', 'cognitive_dysfunction', 0.30)
    s('joint_pain', 'joint_hypermobility', 0.35)
    s('sleep_disturbance', 'cognitive_dysfunction', 0.30)
    s('fatigue', 'gi_symptoms', 0.20)
    s('chronic_pain', 'headaches', 0.25)
    s('anxiety', 'gi_symptoms', 0.20)
    s('skin_fragility', 'subluxations', 0.15)
    s('urinary_symptoms', 'gi_symptoms', 0.20)
    s('tmj_dysfunction', 'headaches', 0.25)
    s('vision_issues', 'headaches', 0.20)
    s('hearing_tinnitus', 'tmj_dysfunction', 0.20)
    eigvals, eigvecs = np.linalg.eigh(corr)
    eigvals = np.maximum(eigvals, 1e-6)
    corr = eigvecs @ np.diag(eigvals) @ eigvecs.T
    np.fill_diagonal(corr, 1.0)
    return corr

CORR_MATRIX = build_correlation_matrix()

SUBTYPE_DEFS = {
    'Musculoskeletal': {
        'proportion': 0.40,
        'elevated': {'joint_pain': 1.2, 'subluxations': 1.3, 'joint_hypermobility': 0.8,
                     'chronic_pain': 0.6, 'skin_fragility': 0.5}
    },
    'Autonomic': {
        'proportion': 0.35,
        'elevated': {'pots_dysautonomia': 1.4, 'fatigue': 1.0, 'cognitive_dysfunction': 0.8,
                     'sleep_disturbance': 0.7, 'anxiety': 0.5}
    },
    'Neurological': {
        'proportion': 0.25,
        'elevated': {'headaches': 1.3, 'cognitive_dysfunction': 1.0, 'vision_issues': 1.2,
                     'tmj_dysfunction': 0.8, 'hearing_tinnitus': 0.7}
    }
}

print('Subtype definitions loaded.')

## 2. Generate Synthetic Data WITH Known Subtypes

In [ ]:
def generate_known_structure_data(n_patients=5000, seed=42):
    np.random.seed(seed)
    n_vars = len(SYMPTOM_NAMES)
    name_to_idx = {name: i for i, name in enumerate(SYMPTOM_NAMES)}
    L = cholesky(CORR_MATRIX, lower=True)

    subtype_labels = []
    subtype_ids = []
    all_latent = []

    subtypes = list(SUBTYPE_DEFS.keys())
    proportions = [SUBTYPE_DEFS[s]['proportion'] for s in subtypes]
    n_per_subtype = np.random.multinomial(n_patients, proportions)

    for sub_idx, (subtype_name, sub_def) in enumerate(SUBTYPE_DEFS.items()):
        n_sub = n_per_subtype[sub_idx]
        means = np.zeros(n_vars)
        for symptom, shift in sub_def['elevated'].items():
            means[name_to_idx[symptom]] = shift
        z = np.random.randn(n_sub, n_vars)
        latent = z @ L.T + means
        all_latent.append(latent)
        subtype_labels.extend([subtype_name] * n_sub)
        subtype_ids.extend([sub_idx] * n_sub)

    latent_all = np.vstack(all_latent)

    # Threshold to ordinal (0=absent, 1=mild, 2=severe)
    ordinal = np.zeros((n_patients, n_vars), dtype=int)
    for i, name in enumerate(SYMPTOM_NAMES):
        prev = PREVALENCES[name]
        t_absent = norm.ppf(1 - prev)
        t_severe = norm.ppf(1 - prev * 0.40)
        ordinal[:, i] = np.where(
            latent_all[:, i] < t_absent, 0,
            np.where(latent_all[:, i] < t_severe, 1, 2)
        )

    df = pd.DataFrame(ordinal, columns=SYMPTOM_NAMES)
    df['subtype_true'] = subtype_labels
    df['subtype_id'] = subtype_ids
    perm = np.random.permutation(n_patients)
    df = df.iloc[perm].reset_index(drop=True)
    return df, perm

df, perm = generate_known_structure_data(n_patients=5000, seed=42)
print(f'Generated {len(df)} patients')
print(df['subtype_true'].value_counts().to_string())

## 3. Convert Ordinal Data to Probabilities for OrdinalSustain

OrdinalSustain expects `prob_nl` (probability of normal) and `prob_score` (probability of each score level), not raw ordinal data.

In [ ]:
data = df[SYMPTOM_NAMES].values.astype(int)
n_subjects = data.shape[0]
n_biomarkers = data.shape[1]
n_scores = N_SCORES  # 2 severity levels above normal

# Probability distributions assuming 90% measurement accuracy
p_correct = 0.9
p_nl_dist = np.full((n_scores + 1), (1 - p_correct) / n_scores)
p_nl_dist[0] = p_correct  # If score=0 (absent), high prob of being normal

p_score_dist = np.full((n_scores, n_scores + 1), (1 - p_correct) / n_scores)
for score in range(n_scores):
    p_score_dist[score, score + 1] = p_correct

# Convert raw ordinal to probability arrays
prob_nl = np.zeros((n_subjects, n_biomarkers))
prob_score = np.zeros((n_subjects, n_biomarkers, n_scores))
for i in range(n_subjects):
    for b in range(n_biomarkers):
        score = data[i, b]
        prob_nl[i, b] = p_nl_dist[score]
        for z in range(n_scores):
            prob_score[i, b, z] = p_score_dist[z, score]

score_vals = np.tile(np.arange(1, n_scores + 1), (n_biomarkers, 1))

print(f'prob_nl shape: {prob_nl.shape}')
print(f'prob_score shape: {prob_score.shape}')
print(f'score_vals shape: {score_vals.shape}')
print(f'N_patients: {n_subjects}')

## 4. Fit GPU-Accelerated Ordinal SuStaIn

Using `TorchOrdinalSustain` with GPU (25x faster than CPU). Fitting k=1 to k=4 with 10,000 MCMC iterations.

In [ ]:
from pySuStaIn.TorchOrdinalSustain import TorchOrdinalSustain

output_dir = tempfile.mkdtemp(prefix='sustain_known_structure_')
print(f'Output: {output_dir}')

sustain = TorchOrdinalSustain(
    prob_nl, prob_score, score_vals, SYMPTOM_NAMES,
    N_startpoints=10,
    N_S_max=4,
    N_iterations_MCMC=10000,
    output_folder=output_dir,
    dataset_name='known_structure',
    use_parallel_startpoints=False,
    seed=42,
    use_gpu=USE_GPU,
    device_id=0
)

print('TorchOrdinalSustain initialized.')

In [ ]:
t0 = time.time()
samples_sequence, samples_f, ml_subtype, prob_ml_subtype, \
ml_stage, prob_ml_stage, prob_subtype_stage = sustain.run_sustain_algorithm()
elapsed = time.time() - t0

# Squeeze from (n,1) to (n,)
ml_subtype = np.squeeze(ml_subtype).astype(int)
ml_stage = np.squeeze(ml_stage)
prob_ml_subtype = np.squeeze(prob_ml_subtype)

print(f'\nFitting complete in {elapsed/60:.1f} minutes')
print(f'Subtypes found: {len(np.unique(ml_subtype))}')
for st in sorted(np.unique(ml_subtype)):
    n = np.sum(ml_subtype == st)
    print(f'  Subtype {int(st)}: {n} patients ({n/len(ml_subtype):.1%})')

## 5. Assess Recovery via Adjusted Rand Index

In [ ]:
ground_truth = df['subtype_id'].values
ari = adjusted_rand_score(ground_truth, ml_subtype)

print('=' * 60)
print('KNOWN-STRUCTURE RECOVERY')
print('=' * 60)
print(f'Adjusted Rand Index:  {ari:.3f}')
print(f'Success threshold:    ARI > 0.7')
print(f'Result:               {"PASS" if ari > 0.7 else "FAIL"}')
print('=' * 60)

In [ ]:
# Hungarian algorithm for best label mapping
n_found = len(np.unique(ml_subtype))
n_true = len(np.unique(ground_truth))
cost = np.zeros((n_found, n_true))
for i in range(n_found):
    for j in range(n_true):
        cost[i, j] = -np.sum((ml_subtype == i) & (ground_truth == j))

row_ind, col_ind = linear_sum_assignment(cost)
mapping = dict(zip(row_ind, col_ind))

print('Best label mapping (SuStaIn -> Ground Truth):')
for r, c in zip(row_ind, col_ind):
    matched = np.sum((ml_subtype == r) & (ground_truth == c))
    total = np.sum(ml_subtype == r)
    pct = matched/total if total > 0 else 0
    print(f'  SuStaIn {r} -> {SUBTYPE_NAMES_TRUE[c]}  ({matched}/{total} = {pct:.1%})')

# Remapped accuracy
remapped = np.zeros_like(ml_subtype)
for old, new in mapping.items():
    remapped[ml_subtype == old] = new
accuracy = np.mean(remapped == ground_truth)
print(f'\nRemapped classification accuracy: {accuracy:.1%} (chance=33%)')

## 6. Evaluate Against Pre-Specified Validation Criteria

In [ ]:
# Criterion 1: Prevalence >= 10%
min_prev = min(np.sum(ml_subtype == s) / len(ml_subtype) for s in np.unique(ml_subtype))
c1_pass = min_prev >= 0.10
print(f'Criterion 1 (Prevalence >= 10%): {"PASS" if c1_pass else "FAIL"} (min: {min_prev:.1%})')

# Criterion 2: Distinctiveness
n_domains_passing = 0
print('\nCriterion 2 (Distinctiveness by domain):')
for domain_name, symptoms in DOMAINS.items():
    max_d = 0
    for s1, s2 in combinations(np.unique(ml_subtype), 2):
        d1 = df.loc[ml_subtype == s1, symptoms].mean(axis=1)
        d2 = df.loc[ml_subtype == s2, symptoms].mean(axis=1)
        pooled = np.sqrt((d1.var() + d2.var()) / 2)
        d = abs(d1.mean() - d2.mean()) / pooled if pooled > 0 else 0
        max_d = max(max_d, d)
    passes = max_d >= 0.5
    if passes:
        n_domains_passing += 1
    print(f'  {domain_name:<20} max d = {max_d:.3f} {"PASS" if passes else "FAIL"}')
c2_pass = n_domains_passing >= 3
print(f'\nCriterion 2: {"PASS" if c2_pass else "FAIL"} ({n_domains_passing}/7 domains)')

# Criterion 5: Not severity-only
total_burden = df[SYMPTOM_NAMES].sum(axis=1)
r_severity = np.corrcoef(ml_subtype, total_burden)[0, 1]
c5_pass = abs(r_severity) < 0.5
print(f'\nCriterion 5 (Not severity-only): {"PASS" if c5_pass else "FAIL"} (r = {r_severity:.3f})')

## 7. Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)

# Ground truth
for i, st_name in enumerate(SUBTYPE_NAMES_TRUE):
    mask = df['subtype_true'] == st_name
    means = df.loc[mask, SYMPTOM_NAMES].mean()
    axes[0].barh(range(N_BIOMARKERS), means, alpha=0.6, label=st_name)
axes[0].set_yticks(range(N_BIOMARKERS))
axes[0].set_yticklabels(SYMPTOM_NAMES, fontsize=9)
axes[0].set_xlabel('Mean Severity (0-2)')
axes[0].set_title('Ground Truth Subtype Profiles')
axes[0].legend(fontsize=9, loc='lower right')
axes[0].invert_yaxis()

# SuStaIn-recovered
for s in sorted(np.unique(ml_subtype)):
    mask = ml_subtype == s
    means = df.loc[mask, SYMPTOM_NAMES].mean()
    label = SUBTYPE_NAMES_TRUE[mapping[int(s)]] if int(s) in mapping else f'Subtype {int(s)}'
    axes[1].barh(range(N_BIOMARKERS), means, alpha=0.6, label=f'SuStaIn {int(s)} ({label})')
axes[1].set_xlabel('Mean Severity (0-2)')
axes[1].set_title('SuStaIn-Recovered Subtype Profiles')
axes[1].legend(fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('known_structure_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(ground_truth, remapped)
sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
            xticklabels=SUBTYPE_NAMES_TRUE,
            yticklabels=SUBTYPE_NAMES_TRUE)
ax.set_xlabel('Predicted Subtype')
ax.set_ylabel('True Subtype')
ax.set_title(f'Recovery Confusion Matrix\nARI = {ari:.3f}, Accuracy = {accuracy:.1%}')
plt.tight_layout()
plt.savefig('known_structure_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Results

In [ ]:
results = {
    'runtime_minutes': float(elapsed/60),
    'gpu_used': bool(USE_GPU),
    'ari': float(ari),
    'remapped_accuracy': float(accuracy),
    'true_k': 3,
    'n_subtypes_found': int(len(np.unique(ml_subtype))),
    'criterion_1_pass': bool(c1_pass),
    'criterion_2_pass': bool(c2_pass),
    'criterion_2_domains_passing': int(n_domains_passing),
    'criterion_5_pass': bool(c5_pass),
    'criterion_5_severity_r': float(r_severity),
}

with open('known_structure_recovery_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Results saved.')
print(json.dumps(results, indent=2))